**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Distributed Training II

Where [Scale_NN's](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) 'GPU toolbox map' becomes runnable: real multi-process training with torch.distributed (the CPU `gloo` backend — same API as multi-GPU NCCL), the all-reduce that powers it, and the sharding arithmetic of ZeRO/FSDP. The data-parallel gradient is verified equal to the single-process big batch.

## 1. Pre-requisites

[Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb), [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) (processes!). Runs on any multi-core machine — the backend swaps to `nccl` unchanged on a GPU cluster.

---
### 🕐 Session 1 of 3 — *Collectives: All-Reduce & Friends* (~35 min)
**Goal:** the communication primitives; ring all-reduce's bandwidth optimality, derived.
**Builds on:** [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 2 (data parallelism, verified).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Collectives — All-Reduce & Friends</b></summary>

**Timing (~35 min).** 10 min the collective vocabulary · 12 min the ring derivation · 13 min running four processes.

**Open with the reduction that makes distributed training teachable.** All of it — DDP, ZeRO, FSDP, pipeline parallelism — is built from a handful of **collectives**: broadcast (one → all), all-reduce (everyone ends with the sum), all-gather, reduce-scatter. **Gradient averaging *is* one all-reduce per step.** Once the room owns those four verbs, distributed-training papers become readable.

**Derive the ring's optimality rather than asserting it; the arithmetic is short and it is the session's one real result.** The naive scheme sends everything to a chief and back — $2(N{-}1)$ copies through **one link**, so it gets worse as workers are added. Ring all-reduce splits the tensor into $N$ chunks and passes them round: reduce-scatter for $N{-}1$ steps, all-gather for $N{-}1$ steps, **every link busy throughout**. Traffic per node is $2\frac{N-1}{N}$ times the data — **which approaches 2 and never exceeds it, independent of $N$.**

**Say what that independence buys, because it is the whole reason the technique matters.** Going from 8 workers to 800 does **not** increase per-node communication. **Bandwidth cost is constant in cluster size**, and that single property is what makes thousand-GPU training possible.

**Then the practical framing of the demo.** `gloo` is the CPU backend; `nccl` is the GPU one; **the API is identical and the swap is one word** on `init_process_group`. That is a genuine pedagogical gift — the workshop runs on a laptop and the code transfers to a cluster unmodified.

**Walk the boilerplate at the top of `dist_worker.py`, since every distributed script begins this way.** `MASTER_ADDR`/`MASTER_PORT` are the rendezvous point; `rank` is who you are; `world_size` is how many exist. **`init_process_group` blocks until all `world_size` processes arrive** — which is why a crashed worker shows up as a *hang* rather than an error, and why a stale process holding port 29531 is the most common re-run failure.

**Point at the two lines that make the demo's check meaningful.** `torch.manual_seed(0)` **before** constructing the model, so every worker starts from **identical weights** — averaging gradients across models that disagree would mean nothing. And the explicit `Generator().manual_seed(...)` for the data, so all four workers build the *same* 64-sample set before slicing. **Distributed correctness is mostly about controlling exactly what is shared.**

**Prepare the room to read the two output lines as different kinds of claim.** `[all_reduce] → [10, 10, 10, 10]` is a **functional** check: $1+2+3+4=10$, present on every worker, which is what "all"-reduce means. The `8.94e-08` line is a **mathematical claim verified numerically**, and it is the more important of the two — Session 2 is built entirely on it.

**Close by flagging the process model, since it surprises people from a threading background.** These are **separate OS processes** — separate interpreters, separate memory, no shared state, communicating only through explicit collectives. **The GIL is irrelevant here**, which is why `subprocess.Popen` is the launcher. Point back at [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb): the process/thread distinction, with real stakes.
</details>

## 2. The Vocabulary of Together

💡 **Intuition.** Distributed training reduces to a handful of **collectives**: broadcast (one → all), all-reduce (everyone ends with the sum), all-gather, reduce-scatter. The star is all-reduce — gradient averaging *is* one all-reduce per step. The naive way (everyone sends to a chief, chief sends back) moves $2(N{-}1)$ copies of the data through one link. **Ring all-reduce** pipelines chunks around a ring: every link busy, total traffic per node $2\frac{N-1}{N}\times$ the data — *independent of N*. That factor is why thousand-GPU training is possible at all.

In [1]:
%%writefile dist_worker.py
# the worker script each process runs — the pattern for ALL torch.distributed work
import torch, torch.nn as nn, torch.distributed as dist, os, sys

rank, world = int(sys.argv[1]), int(sys.argv[2])
os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("MASTER_PORT", "29531")
dist.init_process_group("gloo", rank=rank, world_size=world)     # "nccl" on GPUs — only change

# --- demo 1: all_reduce ---
t = torch.ones(4) * (rank + 1)
dist.all_reduce(t)                                               # in-place sum across workers
if rank == 0:
    print(f"[all_reduce] {world} workers → {t.tolist()}  (= sum 1..{world} in every slot)")

# --- demo 2: data-parallel gradient == big-batch gradient ---
torch.manual_seed(0)                                             # SAME init on every worker
model = nn.Linear(8, 1)
full_x = torch.randn(64, 8, generator=torch.Generator().manual_seed(1))
full_y = torch.randn(64, 1, generator=torch.Generator().manual_seed(2))
shard = slice(rank*64//world, (rank+1)*64//world)                # each worker: its shard

loss = ((model(full_x[shard]) - full_y[shard])**2).mean()
loss.backward()
for p in model.parameters():                                     # DDP-by-hand: average gradients
    dist.all_reduce(p.grad)
    p.grad /= world

if rank == 0:
    ref = nn.Linear(8, 1)
    torch.manual_seed(0); ref = nn.Linear(8, 1)                  # identical init
    ((ref(full_x) - full_y)**2).mean().backward()
    gap = max((p.grad - q.grad).abs().max().item()
              for p, q in zip(model.parameters(), ref.parameters()))
    print(f"[DDP oracle] max |sharded-averaged grad − big-batch grad| = {gap:.2e}")
dist.destroy_process_group()

Writing dist_worker.py


In [2]:
import subprocess, sys
procs = [subprocess.Popen([sys.executable, "dist_worker.py", str(r), "4"],
                          stdout=subprocess.PIPE, text=True) for r in range(4)]
for p in procs:
    out, _ = p.communicate(timeout=120)
    if out: print(out, end="")

[all_reduce] 4 workers → [10.0, 10.0, 10.0, 10.0]  (= sum 1..4 in every slot)
[DDP oracle] max |sharded-averaged grad − big-batch grad| = 8.94e-08


**What just happened.** Four independent OS processes started, found each other over a socket, and produced two lines:

```
[all_reduce] 4 workers → [10.0, 10.0, 10.0, 10.0]
[DDP oracle] max |sharded-averaged grad − big-batch grad| = 8.94e-08
```

**The first is a functional check, and it reads exactly as designed.** Worker $r$ contributed a tensor of $(r+1)$, and $1+2+3+4 = 10$ — appearing in **every slot on every worker**. Not a gather to rank 0, not a reduction to one place: **everyone ends holding the sum**, which is what the "all" means.

**The second is the one that matters, and it is a mathematical claim verified numerically.** $8.94\times10^{-8}$ sits at float32 machine epsilon ($\varepsilon \approx 1.2\times10^{-7}$). **Sharding a 64-sample batch across four workers and averaging their gradients gives the same gradient as computing the whole batch in one place** — not approximately, but to the limit of the arithmetic.

**That identity is the entire correctness story of data-parallel training, and it is one line of algebra.**

$$\nabla \Big(\tfrac{1}{64}\textstyle\sum_{i=1}^{64}\ell_i\Big) \;=\; \tfrac{1}{4}\sum_{r=0}^{3} \nabla\Big(\tfrac{1}{16}\textstyle\sum_{i \in \text{shard}_r}\ell_i\Big)$$

**Differentiation is linear, so splitting the sum changes nothing.** It is precisely the gradient-accumulation argument from [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) Session 2 — the same theorem with a network between the terms instead of a `for` loop.

**Note that the residual is *not* zero, and that this is the right outcome.** Four partial sums combined in a different order from one long sum produce different rounding, because **floating-point addition is not associative**. $10^{-8}$ on gradients of order 1 is exactly what float32 predicts. **An exact 0.0 would be more suspicious**, since it would suggest the two paths were not genuinely independent.

**Two lines in the worker script are what make the check meaningful; point at them.** `torch.manual_seed(0)` before building the model gives all four workers **identical initial weights** — averaging gradients from models that disagree is meaningless. And the explicit `Generator().manual_seed(...)` means each worker constructs the *same* 64 samples before taking its own slice. **Distributed correctness is mostly about controlling precisely what is shared and what is not.**

**Note also what this hand-rolled version leaves out, because real DDP adds exactly one thing.** Here every gradient is all-reduced *after* the whole backward pass finishes. Production DDP all-reduces **bucket by bucket as backprop produces each gradient**, overlapping communication with the remaining backward computation — **the streams idea from [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb), at cluster scale.** Same mathematics, better choreography.

**And note the wall that choreography eventually hits.** Per-step communication is roughly **2 bytes per parameter** (fp16 gradients), independent of batch size. Shrink the step time enough — more workers, smaller per-worker batch — and transfer time exceeds compute time, at which point **adding workers stops helping**. That is the [roofline](./Performance_Engineering.ipynb) argument with a network on the memory axis.

**Finally, the practical note for re-running this cell.** `init_process_group` **blocks until all four processes arrive**, so a worker that crashes early makes the others hang rather than fail. A timeout here usually means a stale process still holds port 29531. **A hang, not an exception, is the characteristic failure of collective code** — worth meeting on a laptop rather than on a cluster.

---
### 🕐 Session 2 of 3 — *Data Parallelism, Verified* (~35 min)
**Goal:** what DDP actually does per step; the overlap trick; when communication becomes the wall.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sharding: ZeRO/FSDP).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Data Parallelism, Verified</b></summary>

**Timing (~35 min).** 10 min what the oracle proved · 12 min the overlap trick · 13 min the communication wall.

**Open by naming what Session 1 established, because it is unusually strong for a systems topic.** Not "DDP works well" but **"DDP is exactly equivalent to a big batch, verified to $8.9\times10^{-8}$."** That is a mathematical claim with a numerical receipt. **Most distributed-systems teaching offers plausibility; this offers a proof and a measurement**, and the room should be told that explicitly.

**Then have them state the theorem in one line, because it is the whole content.** The gradient of a mean is the mean of the gradients — differentiation is linear, so splitting a sum across machines changes nothing. **DDP is gradient accumulation with a network between the terms**, which is precisely the identity [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) Session 2 proved for micro-batches on one device.

**Emphasise the two conditions that make it exact, since both are broken in practice by beginners.** Every worker must start from **identical weights** (hence `torch.manual_seed(0)` before construction), and the shards must be **equal in size** so a plain average is the right weighting. **Unequal shards with uniform averaging silently reweight your data** — the same bug as ragged micro-batches in gradient accumulation.

**Now the one thing real DDP adds, and frame it as choreography rather than mathematics.** The hand-rolled version all-reduces *after* the backward pass completes. Production DDP fires an all-reduce for each **bucket of gradients as backprop produces it**, so communication for early layers overlaps with computation of later ones. **Nothing about the result changes; the waiting disappears** — exactly the streams idea from [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb), one level up.

**Then the wall, with the arithmetic, because this is where scaling actually stops.** Per step, each worker communicates about **2 bytes per parameter** (fp16 gradients) — and that cost is **independent of batch size**. Compute per step scales with the local batch; communication does not. **Add enough workers, shrink the per-worker batch enough, and transfer time exceeds compute time**, at which point more machines buy nothing.

**Make it concrete with one calculation.** A 1B-parameter model moves ~2 GB per step per worker. On a 100 GB/s interconnect that is 20 ms of communication — so **if a step takes less than 20 ms of compute, you are network-bound.** That is the [roofline](./Performance_Engineering.ipynb) with a network on the memory axis, and it explains why frontier training uses large per-worker batches and why interconnect bandwidth, not FLOPs, is often the binding specification of a cluster.

**Draw the consequence for how people should read scaling claims.** "We scaled to 1,024 GPUs" says nothing without **efficiency** — the fraction of ideal speedup retained. **A 1,024-GPU run at 40% efficiency is a 410-GPU run with a larger electricity bill.** Ask the room what they would need to see reported; the answer is throughput per worker, not aggregate throughput.

**Close by pointing at the sibling application, since it reframes the whole session.** [Federated Learning](../Intro_Mach_Learn/Federated_Learning_Privacy.ipynb) runs the same averaging for a completely different reason — privacy rather than speed — and hits a different wall: **non-IID data**, where clients hold systematically different shards and the average of specialists is not a generalist. **Same collective, opposite constraint**, and comparing the two is the fastest way to see what averaging does and does not guarantee.
</details>

## 3. What You Just Proved

💡 **Intuition.** The oracle line above is the entire correctness story of DDP: sharding the batch and averaging gradients is *mathematically identical* to one big batch ([gradient accumulation](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb)'s twin, across machines). Real DDP adds one performance trick: gradients are all-reduced **bucket by bucket as backprop produces them**, overlapping communication with the rest of the backward pass — the [streams idea](./HW_Accelerated_Computing.ipynb) at cluster scale. The wall: per-step communication is ~2 bytes/param (fp16 grads); when step time shrinks below transfer time, scaling stalls — the [roofline](./Performance_Engineering.ipynb) with a network axis.

---
### 🕐 Session 3 of 3 — *Sharding: the ZeRO/FSDP Arithmetic* (~30 min)
**Goal:** when the MODEL doesn't fit: shard optimizer states, gradients, weights — the memory ladder.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Sharding — the ZeRO/FSDP Arithmetic</b></summary>

**Timing (~30 min).** 8 min the problem DDP cannot solve · 12 min the ladder · 10 min the communication cost and the exercise.

**Open by naming the limitation Sessions 1–2 leave untouched, because it is easy to miss.** DDP splits the **batch**. Every worker still holds a **complete copy** of the model, its gradients, and its optimiser states. **So DDP solves "this is too slow" and does nothing at all for "this does not fit."** That distinction is the entire motivation for sharding, and rooms often conflate the two.

**Recover the 16 bytes/param from [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) by having the room rebuild it rather than reading it.** fp32 weights (4) + fp32 gradients (4) + Adam's two moment estimates (8) = **16 bytes per parameter**, before any activations. **A 7B model therefore needs ~112 GB just for training state** — more than any single accelerator. That number is the reason ZeRO exists.

**Then walk the ladder as a sequence of answers to "what else can be split?"** ZeRO-1 shards the **optimiser states** (the largest and least frequently touched piece). ZeRO-2 adds **gradients**. ZeRO-3 / FSDP adds the **weights** themselves. **Each rung shards something that was previously replicated $N$ times for no reason.**

**Have the room verify one row of the table, because the arithmetic is instructive.** ZeRO-2 at $N = 8$: gradients and optimiser states are sharded (12 bytes ÷ 8 = 1.5), weights are not (4) — total **5.5 bytes/param**. **The unsharded piece dominates as $N$ grows**, which is exactly why ZeRO-3 exists and why the ladder does not stop at 2.

**Make the cost explicit, since the table shows only the benefit.** **Every rung buys memory with communication.** ZeRO-3 must **all-gather each layer's weights just-in-time** for its forward pass, use them, and immediately drop them — then do it again on the backward pass. **The parameters stream through each worker rather than living there.** The text's overlap-save analogy is exact: block-stream something too large to hold, keeping only what the current block needs.

**So state the trade as the practical rule.** ZeRO-1 is nearly free and should almost always be on. **ZeRO-3 roughly doubles communication volume** and pays off only when the model genuinely does not fit — which is precisely the [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) discipline of finding the binding constraint before spending anything on it.

**Name the other two axes briefly, so "3-D parallelism" stops being jargon.** **Pipeline parallelism** splits by *layer* (worker 1 holds layers 1–8, worker 2 holds 9–16) and introduces bubble overhead. **Tensor parallelism** splits *within* a matmul and needs very fast interconnect. **Data, pipeline, and tensor are three orthogonal axes**, and frontier runs use all three at once — which is all the phrase means.

**Set the exercise properly, because its verification step is the point.** Extending `dist_worker.py` to ZeRO-1 is genuinely instructive, and the instruction that matters is the last one: **verify the training curves match plain DDP.** A sharding bug does not crash — it produces a model that trains slightly worse, silently. **The oracle discipline from Session 1 is what catches it**, and carrying that habit into a memory optimisation is the real deliverable of this workshop.
</details>

## 4. The Memory Ladder

[Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) counted Adam-fp32 training at ≈16 bytes/param. Sharding across $N$ workers divides what it can:

| Stage | Shards | bytes/param/worker (N=8) |
|---|---|---|
| DDP (none) | — | 16 |
| ZeRO-1 | optimizer states (8B) | 8 + 8/8 = 9 |
| ZeRO-2 | + gradients (4B) | 4 + 12/8 = 5.5 |
| ZeRO-3 / FSDP | + weights (4B) | 16/8 = 2 |

💡 **Intuition.** Each rung trades memory for *communication choreography*: ZeRO-3 must all-gather each layer's weights just-in-time for its forward/backward, then drop them — streaming the model through workers the way [overlap-save](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) streams a signal through a filter. Pipeline and tensor parallelism split along different axes (layers; within-matmul) and compose with all of the above — the '3-D parallelism' of frontier training runs.

**Exercise:** extend `dist_worker.py` to ZeRO-1 — keep Adam moments for only your shard of parameters, all-gather updated weights after each step, and verify training curves match plain DDP.

---
## Where next

- [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the single-node accounting.
- [Federated Learning](../Intro_Mach_Learn/Federated_Learning_Privacy.ipynb) — averaging for privacy instead of speed.
- [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb) — the intra-GPU story.